# Notebook 10 - Battery Sentinel : synthèse finale

Ce notebook relie les deux parties du projet.

On y retrouve :
- un rappel simple des résultats de la partie diffusion ;
- une lecture des résultats Battery Sentinel ;
- une figure finale qui rassemble les deux couches du projet.


## 1. préparation

Cette section charge les bibliothèques utiles et prépare l'accès aux résultats déjà calculés.


In [ ]:
!pip install -q matplotlib pandas numpy pillow


In [ ]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception:
    drive = None

FOUNDATION_DRIVE_DIR = '/content/drive/MyDrive/diffusion_noise_project'
FOUNDATION_DIR = '/content/foundation_diffusion_noise_project'
BATTERY_ROOT = '/content/drive/MyDrive/battery_sentinel'

if IN_COLAB:
    if not os.path.exists(FOUNDATION_DIR):
        if not os.path.exists(FOUNDATION_DRIVE_DIR):
            raise FileNotFoundError(f'Foundation artifacts not found in Drive: {FOUNDATION_DRIVE_DIR}')
        shutil.copytree(FOUNDATION_DRIVE_DIR, FOUNDATION_DIR)
else:
    FOUNDATION_DIR = str((Path.cwd() / 'diffusion_noise_project').resolve())
    BATTERY_ROOT = str((Path.cwd() / 'battery_sentinel').resolve())

LOG_DIR = os.path.join(BATTERY_ROOT, 'logs')
FIGURE_DIR = os.path.join(BATTERY_ROOT, 'figures')
BASE_FIGURE_DIR = os.path.join(FOUNDATION_DIR, 'figures')

print('Partie 1 :', FOUNDATION_DIR)
print('Battery Sentinel :', BATTERY_ROOT)


## 2. Chargement des résultats

On affiche ici les indicateurs principaux de la première partie, puis ceux de Battery Sentinel.


In [ ]:
base_eval = json.load(open(os.path.join(FOUNDATION_DIR, 'logs', 'evaluation_summary.json'), 'r', encoding='utf-8'))
sim_summary = json.load(open(os.path.join(LOG_DIR, 'simulation_summary.json'), 'r', encoding='utf-8'))
twin_summary = json.load(open(os.path.join(LOG_DIR, 'twin_training_summary.json'), 'r', encoding='utf-8'))
router_metrics = json.load(open(os.path.join(LOG_DIR, 'router_metrics.json'), 'r', encoding='utf-8'))

base_rows = []
for noise_type, metrics in base_eval['metrics'].items():
    base_rows.append({
        'noise_type': noise_type,
        'final_epoch_loss': metrics['final_epoch_avg_loss'],
        'avg_denoising_mse': metrics['avg_denoising_mse'],
        'sample_variance': metrics['sample_variance'],
    })

display(pd.DataFrame(base_rows))
display(pd.DataFrame(twin_summary['test_metrics']))


## 3. Lecture simple des résultats

Points à retenir :
- la première partie montre que les trois régimes de bruit ne se comportent pas pareil ;
- Battery Sentinel reprend cette idée dans un cas de monitoring batterie ;
- le **predictive twin** apprend le comportement nominal ;
- le **router** transforme ensuite les écarts en décision interprétable.


## 4. Tableau de bord final

La figure suivante rassemble la partie diffusion et la partie Battery Sentinel dans une seule vue.


In [ ]:
fig = plt.figure(figsize=(15, 10))
grid = fig.add_gridspec(2, 2)

ax1 = fig.add_subplot(grid[0, 0])
ax1.imshow(Image.open(os.path.join(BASE_FIGURE_DIR, 'summary_figure.png')))
ax1.set_title('Partie 1 : Étude diffusion')
ax1.axis('off')

ax2 = fig.add_subplot(grid[0, 1])
ax2.imshow(Image.open(os.path.join(FIGURE_DIR, 'battery_regime_examples.png')))
ax2.set_title('Régimes simulés')
ax2.axis('off')

ax3 = fig.add_subplot(grid[1, 0])
ax3.imshow(Image.open(os.path.join(FIGURE_DIR, 'battery_twin_training_curves.png')))
ax3.set_title('Predictive twin')
ax3.axis('off')

ax4 = fig.add_subplot(grid[1, 1])
ax4.imshow(Image.open(os.path.join(FIGURE_DIR, 'battery_router_confusion.png')))
ax4.set_title('Router final')
ax4.axis('off')

fig.tight_layout()
dashboard_path = os.path.join(FIGURE_DIR, 'battery_sentinel_dashboard.png')
fig.savefig(dashboard_path, dpi=200, bbox_inches='tight')
plt.show()
print('Figure finale mise à jour.')


## 5. Sorties du notebook

Sorties produites :
- `battery_sentinel/logs/battery_sentinel_summary.json`
- `battery_sentinel/figures/battery_sentinel_dashboard.png`


In [ ]:
final_payload = {
    'project_layers': {
        'foundational_study': 'Notebooks 1 à 6',
        'applied_system': 'Notebooks 7 à 10',
    },
    'battery_sentinel_outputs': {
        'simulation_summary': os.path.join(LOG_DIR, 'simulation_summary.json'),
        'twin_training_summary': os.path.join(LOG_DIR, 'twin_training_summary.json'),
        'router_metrics': os.path.join(LOG_DIR, 'router_metrics.json'),
        'dashboard_figure': os.path.join(FIGURE_DIR, 'battery_sentinel_dashboard.png'),
    },
    'router_macro_f1': router_metrics['macro_f1'],
    'baseline_binary_f1': router_metrics['baseline_binary_f1'],
}
with open(os.path.join(LOG_DIR, 'battery_sentinel_summary.json'), 'w', encoding='utf-8') as handle:
    json.dump(final_payload, handle, indent=2)
final_payload
